In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as spopt
import numpy.ma as ma
from tqdm import tqdm
import pandas as pd
import glob, os

import sys
#Need this command for the _merge function for recursion
sys.setrecursionlimit(10**5)

In [3]:
def master_opener():
    
    #Opening the most recent master_df, dont forget to pull the git repo
    master_df = pd.read_csv("/home/ouldboukattine//git/logs/master_df.csv", sep=",", header=0, index_col=0, \
                            na_values='NA', engine='python')
    
    #Adding in the observed_hours per row
    master_df["observed_hours"] = (master_df["MJD_end_time"] - master_df["MJD_start_time"]) * 24
    
    #Sort just because 
    master_df = master_df.sort_values(by="Obs_Date_String")

    #display(master_df)
    
    #FRB20220912A & F220912
    df_r117 = master_df[(master_df["Source_Name"] == "FRB20220912A") | (master_df["Source_Name"] == "F220912") \
                        | (master_df["Source_Name"] == "R220912") & (master_df["Telescope"] != "Nancay") ]
    
    #Ef, Nt and Mc observed in pr247a/pr248a/pr249a   
    df_r117 = df_r117[(df_r117["Telescope"] == "wb") | (df_r117["Telescope"] == "tr") \
                        | (df_r117["Telescope"] == "stk") | (df_r117["Telescope"] == "o8")]
        
    #Filter out Nancay
    #df_r117 = df_r117[(df_r117["Telescope"] != "Nancay")]
    
    display(df_r117)
    
    tels = set(df_r117["Telescope"].values)
    obs_set= set(df_r117["Central_freq_(MHz)"].values)
    print(tels)
    print(obs_set)
    
    return df_r117

df_r117 = master_opener()

,scan_name,scan_no,Telescope,Source_Name,Obs_Date_String,MJD_start_time,RA_J2000,RA_J2000_(deg),Dec_J2000,Dec_J2000_(deg),...,bytes_in_file_header,MJD_end_time,complete_filename,Are_bytes_signed?,Polarization_order,Backend,Starting_subint,Subints_per_file,Source_Name_nancay,observed_hours
64961,r117_59867_1,no0001,stk,FRB20220912A,2022-10-15T08:48:05.4886,59867.366730,23:09:10.0000,347.291667,48:42:00.0000,48.700000,...,364.0,59867.430808,59867.366.txt,NaN,NaN,Unknown,0.0,0.0,NaN,1.537866
64962,r117_59867_2,no0001,stk,FRB20220912A,2022-10-15T12:36:30.1572,59867.525349,23:09:10.0000,347.291667,48:42:00.0000,48.700000,...,364.0,59867.650855,59867.525.txt,NaN,NaN,Unknown,0.0,0.0,NaN,3.012138
64963,r117_59867_3,no0001,stk,FRB20220912A,2022-10-15T15:37:14.2646,59867.650860,23:09:10.0000,347.291667,48:42:00.0000,48.700000,...,364.0,59867.775814,59867.650.txt,NaN,NaN,Unknown,0.0,0.0,NaN,2.998912
64964,r117_59867_4,no0001,stk,FRB20220912A,2022-10-15T18:37:10.8155,59867.775820,23:09:10.0000,347.291667,48:42:00.0000,48.700000,...,364.0,59867.859356,59867.775.txt,NaN,NaN,Unknown,0.0,0.0,NaN,2.004881
64965,r117_59867_5,no0001,stk,FRB20220912A,2022-10-15T21:18:34.691,59867.887902,23:09:10.0000,347.291667,48:42:00.0000,48.700000,...,364.0,59868.014983,59867.887.txt,NaN,NaN,Unknown,0.0,0.0,NaN,3.049959
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142839,p47094,no0048,wb,R220912,2024-06-13T09:56:53,60474.414502,23:09:04.9000,347.270417,48:42:25.4000,48.707056,...,295.0,60474.424884,p47094_wb_no0048_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249164
142840,p47094,no0049,wb,R220912,2024-06-13T10:12:05,60474.425058,23:09:04.9000,347.270417,48:42:25.4000,48.707056,...,295.0,60474.435440,p47094_wb_no0049_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249164
142841,p47094,no0050,wb,R220912,2024-06-13T10:27:17,60474.435613,23:09:04.9000,347.270417,48:42:25.4000,48.707056,...,295.0,60474.445995,p47094_wb_no0050_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249164
142842,p47094,no0051,wb,R220912,2024-06-13T10:42:29,60474.446169,23:09:04.9000,347.270417,48:42:25.4000,48.707056,...,295.0,60474.456551,p47094_wb_no0051_IFall_vdif_pol2.txt,False,NaN,NaN,NaN,NaN,NaN,0.249164


{'wb', 'o8', 'stk', 'tr'}
{4708.0, 1381.46484375, 1414.0, 1418.0, 332.0, 1271.0, 4664.0, 4926.49}


In [4]:
#dictionary with frequency setups
def freq_dict_maker():
    
    #acces with: freq_dict[332]
    freq_dict = {
        332 : "P",
        1271 : "L18",
        1414 : "L15",
        1418 : "L12",        
        1381.46484375 : "L13",
        4664 : "C5",
        4708 : "C4",        
        4926.49 : "C1"
    }
    
    return freq_dict

In [5]:
def _merge(head, tail):
    if tail == []:
        return head

    a, b = head[-1]
    x, y = tail[0]

    do_merge = b > x
    if do_merge:
        head_ = head[:-1] + [(a, max(b, y))]
        tail_ = tail[1:]
        return _merge(head_, tail_)
    else:
        head_ = head + tail[:1]
        tail_ = tail[1:]
        return _merge(head_, tail_)


def merge_intervals(lst):
    if len(lst) <= 1:
        return lst
    
    #sort on the first element of the tuples for the whole list
    lst = sorted(lst, key=lambda x: x[0])

    return _merge(lst[:1], lst[1:])

def get_overlap_times(df, coln1="MJD_start", coln2="MJD_end"):
    
    # get the right columns
    start = np.array(df["MJD_start_time"])
    end = np.array(df["MJD_end_time"])
    
    # check if input is correct:
    for s, e in zip(start, end):
        if s >= e:
            print("WARNING: start time should not exceed end time")
            
    # create tuple pairs:
    times = [(x, y) for x, y in zip(start, end)]
    times_reduced = merge_intervals(times)
    
    return times_reduced

def calc_overlap_times(times, unit="days"):
    """ Times is a list of tuple pairs, e.g.:
    [(1, 3), (5.5, 6.0)]
    Will return the sum of the differences of the pairs:
    [(1, 3), (5.5, 6.0)] --> (3-1) + (6.0-5.5) = 2 + 0.5 = 2.5 """
    ttotal = 0 # total time
    for timepair in times:
        s, e = timepair[0], timepair[1]
        ttotal += (e - s)
    print(f"The total non overlapping observing time is {ttotal:.2f} {unit} or {(ttotal*24):.2f} hours")

In [6]:
def hours_calc(df_source, total_time=False, activity_phase=False, save_bool=False):
    
    #call the freq dictionary
    freq_dict = freq_dict_maker()
    
    #Conmbine all the pickles files into one big dataframe    
    df_source["total_obs_time_s"] = (df_source["MJD_end_time"] - df_source["MJD_start_time"]) * 24 * 3600
        
    #print setup of observing frequenies
    freqs = df_source['Central_freq_(MHz)'].values
    print(set(freqs))
    
    ##If you want to limit in time:
    if activity_phase == True:        
        
        #### Main R117 campaign -- Observational setup ####
        #59866 - 14-Oct-22 00:00:00
        #59983 - 08-Feb-23 00:00:00
        bound_l = 59866 
        bound_r = 59983         
        
        #### Overlap with FAST -- Comp plot #####
        #59910-59869 -- David code
        #59602 start of wb; 59644 last Stk obs
        #59910
#         bound_l = 59869 
#         bound_r = 59910           
        
        #### Activity phase with ATA #####
        ###Our campaign: 59867 -- 59983
        ###ATA campaign: 59872 -- 59933
        #Begin of Europe observavtions: 59867
        #End of ATA observations: 59933
#         bound_l = 59867
#         bound_r = 59933
        
        print(f"!!Using Activity phase!! begin: {bound_l} | end: {bound_r}")
        df_source = df_source[(df_source['MJD_start_time'] > bound_l) & (df_source['MJD_end_time'] < bound_r)]
    
    #Westerbork
    df_wb = df_source[df_source["Telescope"] == 'wb']
    wb_obs_h = sum(df_wb["total_obs_time_s"].values)/3600
    
    #Torun
    df_tr = df_source[df_source["Telescope"] == 'tr']
    tr_obs_h = sum(df_tr["total_obs_time_s"].values)/3600
    
    #Stockert
    df_stk = df_source[df_source["Telescope"] == 'stk']
    stk_obs_h = sum(df_stk["total_obs_time_s"].values)/3600
    
    #Onsala
    df_o8 = df_source[df_source["Telescope"] == 'o8']
    o8_obs_h = sum(df_o8["total_obs_time_s"].values)/3600
    
    #Print observing hours
    print(f"total obs Wb {wb_obs_h:.2f} h")
    print(f"total obs Tr {tr_obs_h:.2f} h")
    print(f"total obs Stk {stk_obs_h:.2f} h")
    print(f"total obs o8 {o8_obs_h:.2f} h")
    
    total_obs = sum(df_source["total_obs_time_s"].values) / 3600
    print(f"total observing time: {total_obs:.2f} h")
    print("--------------------")
    
    print("-P band-")
    #Pband
    df_r117_pband = df_source[df_source["Central_freq_(MHz)"] == 332.0]
    times_p = get_overlap_times(df_r117_pband)
    ttotal_p = calc_overlap_times(times_p)
    
    print("-C band-")
    df_r117_cband = df_source[(df_source["Central_freq_(MHz)"] == 4664.0) | \
                              (df_source["Central_freq_(MHz)"] == 4926.49) | \
                              (df_source["Central_freq_(MHz)"] == 4708.0)]
    
    times_c = get_overlap_times(df_r117_cband)
    ttotal_c = calc_overlap_times(times_c)
    
    print("-L band-")
    #Lband
    df_r117_lband = df_source[(df_source["Central_freq_(MHz)"] != 332.0) & \
                          (df_source["Central_freq_(MHz)"] != 4664.0) & \
                          (df_source["Central_freq_(MHz)"] != 4708.0) & \
                          (df_source["Central_freq_(MHz)"] != 4926.49)   ]
    times_l = get_overlap_times(df_r117_lband)
    ttotal_l = calc_overlap_times(times_l)
    
    #Calculate the unique hours in the observations
    if total_time == True:
        print("--------------------")
        times = get_overlap_times(df_source)
        ttotal = calc_overlap_times(times)
        print("--------------------")
    
    #Counters for observing time
    total_h_obs = 0
    total_h_obs_lband = 0
    
    #Loop over all entries in the freq_dict
    for obser_freq in freq_dict:

        #Create mask per frequency setup
        df_freq_setup_mask = (df_source['Central_freq_(MHz)'] == obser_freq)
        df_freq_setup = df_source[(df_freq_setup_mask)]
        
        #As a double check; get the set of all telescopes in this setup (should be unique)
        part_telescope = set(df_freq_setup["Telescope"])
        
        #Amount of hours per frequency setup in hours
        df_freq_setup_h = sum(df_freq_setup["total_obs_time_s"].values)/3600
        print(f"{freq_dict[obser_freq]}: time = {df_freq_setup_h:.2f} h -- central_freq {obser_freq} -- {part_telescope}")
        
        #Add to total
        total_h_obs += df_freq_setup_h

        #Add if observation is on Lband
        if freq_dict[obser_freq].startswith("L"):
            total_h_obs_lband += df_freq_setup_h
    
    print(f"Total observing on L-band: {total_h_obs_lband:.2f}")
    print(f"as a double check, the total is over the freq setup is: {total_h_obs:.2f}")
    print("--------------------")
    
    ##Saving the full campaign (activity_phase=True) 59866-59983
    ##Ran on 02-08-2023
    if save_bool == True:
        display(df_source)
        df_source = df_source.sort_values(by=['MJD_start_time'])
        csv_obs = "../dbs/r117_fullcampaign.csv"
        df_source.to_csv(csv_obs, sep=",", header=True, index=True, na_rep='NA')
        print("campaign saved")
        
    return df_source

df_r117_campaign = hours_calc(df_r117, total_time=True, activity_phase=True, save_bool=False)

{4708.0, 1381.46484375, 1414.0, 1418.0, 332.0, 1271.0, 4664.0, 4926.49}
!!Using Activity phase!! begin: 59866 | end: 59983
total obs Wb 803.92 h
total obs Tr 425.59 h
total obs Stk 933.53 h
total obs o8 28.36 h
total observing time: 2191.39 h
--------------------
-P band-
The total non overlapping observing time is 25.31 days or 607.34 hours
-C band-
The total non overlapping observing time is 7.61 days or 182.62 hours
-L band-
The total non overlapping observing time is 48.27 days or 1158.57 hours
--------------------
The total non overlapping observing time is 62.12 days or 1490.95 hours
--------------------
P: time = 607.34 h -- central_freq 332 -- {'wb'}
L18: time = 196.58 h -- central_freq 1271 -- {'wb'}
L15: time = 268.10 h -- central_freq 1414 -- {'tr'}
L12: time = 3.23 h -- central_freq 1418 -- {'tr'}
L13: time = 933.53 h -- central_freq 1381.46484375 -- {'stk'}
C5: time = 40.39 h -- central_freq 4664 -- {'tr'}
C4: time = 113.87 h -- central_freq 4708 -- {'tr'}
C1: time = 28.36

In [7]:
df_r117_campaign = hours_calc(df_r117, total_time=True, activity_phase=False, save_bool=False)

{4708.0, 1381.46484375, 1414.0, 1418.0, 332.0, 1271.0, 4664.0, 4926.49}
total obs Wb 1891.03 h
total obs Tr 1114.23 h
total obs Stk 1464.40 h
total obs o8 28.36 h
total observing time: 4498.02 h
--------------------
-P band-
The total non overlapping observing time is 45.18 days or 1084.39 hours
-C band-
The total non overlapping observing time is 7.61 days or 182.62 hours
-L band-
The total non overlapping observing time is 115.24 days or 2765.69 hours
--------------------
The total non overlapping observing time is 141.98 days or 3407.57 hours
--------------------
P: time = 1084.39 h -- central_freq 332 -- {'wb'}
L18: time = 806.63 h -- central_freq 1271 -- {'wb'}
L15: time = 956.74 h -- central_freq 1414 -- {'tr'}
L12: time = 3.23 h -- central_freq 1418 -- {'tr'}
L13: time = 1464.40 h -- central_freq 1381.46484375 -- {'stk'}
C5: time = 40.39 h -- central_freq 4664 -- {'tr'}
C4: time = 113.87 h -- central_freq 4708 -- {'tr'}
C1: time = 28.36 h -- central_freq 4926.49 -- {'o8'}
Total 